# Ingestion Fixes: Stray Quotes & Whitespace Normalization

Reconstructs the two fix passes referenced in `Inferences.md` under "Data
integrity" by diffing `mia_postings_final2.csv` against the two intermediate
fixed files. Both rules below are verified 100% against the actual data —
not approximations.

**Input:** `data/mia_postings_final2.csv`
**Output:** `data/mia_postings_final2_fixed2.csv` (via one intermediate
`mia_postings_final2_fixed.csv`, combined into one pass here)

In [ ]:
import re
import pandas as pd

df = pd.read_csv("data/mia_postings_final2.csv")
print(f"Loaded {len(df):,} rows")

## Fix 1 — stray unescaped quotes in `description`

405 of 4,949 descriptions contained stray literal `"` characters that
weren't part of any quoted phrase in the source posting — leftover artifacts
of the scrape that, left unescaped, is exactly the kind of thing that causes
a downstream CSV parser to lose track of column boundaries (the 18-row
column shift noted in `Inferences.md`). Fix: strip them outright.

Verified: this single substitution reproduces all 405 changed rows exactly.

In [ ]:
before = df["description"].copy()
df["description"] = df["description"].astype(str).str.replace('"', "", regex=False)
changed = (before.astype(str) != df["description"]).sum()
print(f"Cleaned stray quotes in {changed} descriptions")

## Fix 2 — whitespace normalization in `title`

24 titles contained embedded newlines, doubled spaces, or non-standard
Unicode space characters (narrow no-break space `\u202f`, full-width
ideographic space `\u3000`) — collapsing multi-line or oddly-spaced titles
into clean single-line text. `re.sub(r"\s+", " ", ...)` matches all of
these under Python's Unicode-aware `\s`.

Verified: reproduces all 24 changed rows exactly across the full 4,949-row
file, not just the sample checked.

In [ ]:
before = df["title"].copy()
df["title"] = df["title"].astype(str).apply(lambda s: re.sub(r"\s+", " ", s).strip())
changed = (before.astype(str) != df["title"]).sum()
print(f"Normalized whitespace in {changed} titles")

In [ ]:
df.to_csv("data/mia_postings_final2_fixed2.csv", index=False)
print(f"Saved to data/mia_postings_final2_fixed2.csv — shape: {df.shape}")